# RWC -- Colab driver

The only notebook you open. Run the cells top to bottom; if the session drops,
re-run them and the queue picks up exactly where it left off.

* Checkpoints, metric CSVs and the run manifest live on Drive.
* A `running` run whose heartbeat is >20 min stale is reclaimed automatically.
* Each run prints `SAFE TO DISCONNECT` after every checkpoint.


In [ ]:
#@title RWC driver -- run this cell. It resumes whatever is unfinished.
# SPEC.md section 7: one cell, zero manual intervention after a disconnect.
REPO   = "https://github.com/Nafis878/A-.git"
DRIVE  = "/content/drive/MyDrive/rwc"      # checkpoints + logs live here
LOCAL  = "/content/rwc"                    # hot code + data on local disk
MAX_RUNS = None                            # None = keep going until the grid is done

import os, subprocess, sys, pathlib

from google.colab import drive
drive.mount("/content/drive")
pathlib.Path(DRIVE).mkdir(parents=True, exist_ok=True)

if not os.path.exists(LOCAL):
    subprocess.run(["git", "clone", REPO, LOCAL], check=True)
else:
    subprocess.run(["git", "-C", LOCAL, "pull", "--ff-only"], check=False)
sys.path.insert(0, LOCAL)
os.chdir(LOCAL)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"     {props.total_memory/2**30:.0f} GiB, bf16={torch.cuda.is_bf16_supported()}")


## 1. Sanity gates

Run once after any code change.

In [ ]:
#@title Sanity gates -- these must pass before any run is trusted
# The equivalence test is the one that matters: if the parallel and recurrent
# paths disagree, every downstream number is garbage (SPEC.md section 3).
!python -m pytest tests/test_recurrence_equivalence.py tests/test_influence_groundtruth.py -q


## 2. Work the queue

This is the cell you re-run after a disconnect.

In [ ]:
import os
os.environ["DRIVE"] = DRIVE
args = [] if MAX_RUNS is None else ["--max-runs", str(MAX_RUNS)]
!python scripts/run_queue.py --manifest "$DRIVE/manifest.json" --results-dir "$DRIVE/results" {" ".join(args)}


## 3. Status

In [ ]:
#@title Where the grid stands
!python scripts/run_queue.py --manifest "$DRIVE/manifest.json" --results-dir "$DRIVE/results" --status


## 4. Figures

In [ ]:
#@title Figures for C1, C2 and C3
!python scripts/make_figures.py --results-dir "$DRIVE/results" --out-dir "$DRIVE/figures"
